# LoRA fine-tuning sketch: Qwen3-0.6B on free-tier Colab GPU

**Part of:** [local-llm-quantization-benchmark](https://github.com/Harshal875/local-llm-quantization-benchmark)
— a CPU-only local inference/quantization project. Fine-tuning needs a GPU
that laptop doesn't have, so this notebook is meant to run on **Colab's
free T4 GPU** (Runtime > Change runtime type > T4 GPU) instead.

**Status: sketch, not run/verified.** This is a structured starting point
showing the intended LoRA fine-tuning pipeline with
[unsloth](https://github.com/unslothai/unsloth) — it has not been executed
end-to-end (no GPU available to verify it locally). Expect to debug
version/dependency issues on first run, as with any ML training script
that hasn't been executed yet.

**Goal:** fine-tune Qwen3-0.6B (the same base model used for the CPU
quantization benchmarks in the main repo) on a small instruction dataset,
then export a GGUF so the result can be dropped back into the same
llama.cpp benchmarking scripts used there.

A free Colab T4 GPU has ~15GB VRAM, which is generous for a 0.6B model —
even full fine-tuning would likely fit, but LoRA is used here to keep
training fast and the output adapter small.

## 1. Install dependencies

`unsloth` bundles compatible versions of `torch`, `trl`, `peft`,
`accelerate`, and `bitsandbytes` for the Colab GPU environment.

In [ ]:
!pip install -q unsloth
# If unsloth's pinned deps lag a Colab image update, this often fixes it:
# !pip install -q --upgrade --no-deps --force-reinstall unsloth unsloth_zoo

## 2. Load the base model in 4-bit

Loading in 4-bit (via `bitsandbytes`) keeps VRAM usage low, leaving
headroom for training activations and optimizer state on the free T4.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-0.6B",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,       # auto-detect: bf16 on T4-class GPUs that support it, else fp16
    load_in_4bit=True,
)

## 3. Attach a LoRA adapter

Standard small-model LoRA settings: rank 16, applied to the attention and
MLP projection layers. `gradient_checkpointing="unsloth"` trades a bit of
speed for meaningfully lower VRAM use, which matters on a free-tier GPU.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. Prepare a small instruction dataset

Using a modest subset (1,000 examples) of a standard instruction-tuning
dataset keeps a demo training run short on a free GPU. Swap in a
domain-specific dataset here for a real fine-tune.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned", split="train[:1000]")

def format_example(example):
    messages = [
        {"role": "user", "content": example["instruction"] + ("\n\n" + example["input"] if example["input"] else "")},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_example)

## 5. Train

`max_steps=60` keeps this a short demo run (a few minutes on a T4) rather
than a full training run — raise it (or switch to `num_train_epochs`) for
an actual fine-tune.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

## 6. Save the adapter, and export a merged GGUF

Saves the LoRA adapter alone (small, a few MB) and, separately, a merged
Q4_K_M GGUF — the same quant level used throughout the main benchmark repo
— so the fine-tuned model can be dropped straight into
`local-llm-quantization-benchmark/models/gguf/` and benchmarked with the
existing `scripts/benchmark.py`.

In [ ]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

# Merges the LoRA adapter into the base weights and quantizes straight to
# GGUF via unsloth's bundled llama.cpp conversion path.
model.save_pretrained_gguf("qwen3-0.6b-finetuned", tokenizer, quantization_method="q4_k_m")

## 7. Download the result

Download `qwen3-0.6b-finetuned/*.gguf` from the Colab file browser (or
mount Google Drive and copy it there), then move it into this repo's
`models/gguf/` directory locally to benchmark it with the same
CPU-inference scripts used for the base model.

## Known gaps / next steps

- **Not run end-to-end.** First run will likely need minor fixes —
  package version drift on Colab's default image, exact `target_modules`
  naming for this model, or chat-template formatting details.
- **Dataset is generic** (Alpaca) purely for pipeline demonstration —
  a real fine-tune should use a task-specific dataset.
- **No eval step.** A real run should hold out a validation split and
  track loss (or better, task-specific accuracy) rather than trusting
  training loss alone.
- **`max_steps=60` is a smoke test**, not a real training budget — sized
  to finish in a few minutes to confirm the pipeline works before
  committing to a longer, real fine-tuning run.